# Python reference 12: data engineering patterns

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python/learning/python-reference-12-data-engineering-patterns.ipynb)

Plain-Python versions of the patterns behind reliable pipelines: typed settings, validation with quarantine, schema drift, watermarks and late data, deduplication, content hashes, SCD type 2, retries with jitter, cursor pagination, batch run summaries, backfills, reconciliation and pseudonymisation. Practise it in [Python practice 12: data engineering patterns](../practice/python-practice-12-data-engineering-patterns.ipynb).

*Last reviewed: November 2026, against Python 3.13.*

## Contents

| section | entries |
| --- | --- |
| [Configuration and validation](#configuration-and-validation) | [Typed settings from the environment](#typed-settings-from-the-environment), [Validate records and quarantine rejects](#validate-records-and-quarantine-rejects), [Detect schema drift](#detect-schema-drift) |
| [Incremental and idempotent loads](#incremental-and-idempotent-loads) | [Watermarks and late data](#watermarks-and-late-data), [Deduplicate: keep the latest version](#deduplicate-keep-the-latest-version), [Change detection with content hashes](#change-detection-with-content-hashes), [Slowly changing dimensions (type 2)](#slowly-changing-dimensions-type-2) |
| [Resilient extraction](#resilient-extraction) | [Retries with exponential backoff and jitter](#retries-with-exponential-backoff-and-jitter), [Paginating an API with a cursor](#paginating-an-api-with-a-cursor) |
| [Operating pipelines](#operating-pipelines) | [Batches with bounded memory and a run summary](#batches-with-bounded-memory-and-a-run-summary), [Backfills and reconciliation](#backfills-and-reconciliation), [Pseudonymising personal data](#pseudonymising-personal-data) |

## Configuration and validation

### Typed settings from the environment

Jobs read configuration from environment variables (twelve-factor style), which are always strings. Parse them once,
at start-up, into a typed, frozen object; fail fast with a message that lists **every** problem, not just the first.
In larger projects `pydantic-settings` does this; the standard-library version is a dataclass.

In [ ]:
import os
from dataclasses import MISSING, dataclass, fields

@dataclass(frozen=True)
class Settings:
    source_bucket: str
    batch_size: int = 1000
    dry_run: bool = False
    lookback_days: int = 2

def parse_bool(text):
    if text.lower() in {"1", "true", "yes", "on"}:
        return True
    if text.lower() in {"0", "false", "no", "off"}:
        return False
    raise ValueError(f"not a boolean: {text!r}")

def load_settings(env=os.environ, prefix="JOB_"):
    values, problems = {}, []
    for f in fields(Settings):
        raw = env.get(prefix + f.name.upper())
        if raw is None:
            if f.default is MISSING:   # no default: required
                problems.append(f"{prefix}{f.name.upper()} is required")
            continue
        try:
            values[f.name] = parse_bool(raw) if f.type is bool else f.type(raw)
        except ValueError as e:
            problems.append(f"{prefix}{f.name.upper()}: {e}")
    if problems:
        raise ValueError("; ".join(problems))
    return Settings(**values)

print(load_settings({"JOB_SOURCE_BUCKET": "s3://raw", "JOB_BATCH_SIZE": "500", "JOB_DRY_RUN": "yes"}))
try:
    load_settings({"JOB_BATCH_SIZE": "many", "JOB_DRY_RUN": "maybe"})
except ValueError as e:
    print("ValueError:", e)

### Validate records and quarantine rejects

Never let one bad record fail a whole batch silently or crash it loudly. Validate each record against a list of
rules, pass the good ones on, and write the rejects with their **reasons** to a quarantine location for inspection and
replay. Track the reject rate and fail the run when it crosses a threshold: a sudden spike usually means an upstream
change, not bad luck.

In [ ]:
RULES = [
    ("order_id is required", lambda r: r.get("order_id") is not None),
    ("amount must be a non-negative number", lambda r: isinstance(r.get("amount"), (int, float)) and r["amount"] >= 0),
    ("status is unknown", lambda r: r.get("status") in {"placed", "paid", "shipped"}),
]

def validate(rows, rules=RULES, max_reject_rate=0.5):
    good, rejects = [], []
    for row in rows:
        reasons = [message for message, ok in rules if not ok(row)]
        if reasons:
            rejects.append({"record": row, "reasons": reasons})
        else:
            good.append(row)
    if rows and len(rejects) / len(rows) > max_reject_rate:
        raise RuntimeError(f"reject rate {len(rejects)}/{len(rows)} exceeds {max_reject_rate:.0%}")
    return good, rejects

rows = [{"order_id": 1, "amount": 120.0, "status": "paid"}, {"order_id": None, "amount": -5, "status": "lost"}, {"order_id": 3, "amount": 9, "status": "placed"}]
good, rejects = validate(rows)
print([r["order_id"] for r in good], rejects)

### Detect schema drift

Sources add, drop and retype fields without warning. Compare incoming records with the expected schema before loading,
and decide per case: new nullable fields can be added automatically; missing or retyped fields should stop the load.

In [ ]:
EXPECTED = {"order_id": int, "amount": float, "status": str}

def drift(records, expected=EXPECTED):
    seen = {}
    for r in records:
        for k, v in r.items():
            if v is not None:
                seen.setdefault(k, set()).add(type(v).__name__)
    added = sorted(set(seen) - set(expected))
    missing = sorted(set(expected) - set(seen))
    retyped = sorted(k for k, t in expected.items() if k in seen and seen[k] - {t.__name__, "int" if t is float else t.__name__})
    return {"added": added, "missing": missing, "retyped": retyped}

print(drift([{"order_id": 1, "amount": 9, "status": "paid", "coupon": "X"}, {"order_id": "2", "amount": 1.5, "status": "paid"}]))

## Incremental and idempotent loads

### Watermarks and late data

An incremental job processes only records newer than the last **watermark** it saved (a max timestamp or id), then
stores the new watermark **after** the load commits. Records often arrive late, so re-read a **lookback** window
before the watermark and rely on an idempotent write (upsert or partition overwrite) to absorb the overlap.

| choice | trade-off |
| --- | --- |
| watermark on event time | business-correct, but needs a lookback for late events |
| watermark on ingestion time or an increasing id | no late data issue, but depends on the source providing it |
| lookback window | catches late data at the cost of reprocessing; safe only with idempotent writes |

In [ ]:
from datetime import datetime, timedelta

def incremental(rows, watermark, lookback=timedelta(hours=1)):
    since = watermark - lookback
    batch = [r for r in rows if r["updated_at"] > since]
    new_watermark = max((r["updated_at"] for r in batch), default=watermark)
    return batch, max(new_watermark, watermark)

t = datetime(2026, 11, 27, 9)
rows = [{"id": 1, "updated_at": t - timedelta(hours=3)}, {"id": 2, "updated_at": t - timedelta(minutes=20)}, {"id": 3, "updated_at": t + timedelta(minutes=5)}]
batch, wm = incremental(rows, t)
print([r["id"] for r in batch], wm)

### Deduplicate: keep the latest version

At-least-once delivery means duplicates are normal. Keep one record per business key, choosing the latest by an
update timestamp, with a deterministic tie-breaker (a sequence number, or the ingestion offset) so reruns give the
same answer.

In [ ]:
def latest_per_key(rows, key="id", order=("updated_at", "seq")):
    best = {}
    for r in rows:
        k = r[key]
        if k not in best or tuple(r[o] for o in order) > tuple(best[k][o] for o in order):
            best[k] = r
    return sorted(best.values(), key=lambda r: r[key])

events = [
    {"id": 1, "status": "placed", "updated_at": 1, "seq": 1},
    {"id": 1, "status": "paid", "updated_at": 2, "seq": 2},
    {"id": 2, "status": "placed", "updated_at": 5, "seq": 3},
    {"id": 1, "status": "paid-dup", "updated_at": 2, "seq": 1},
]
print([(r["id"], r["status"]) for r in latest_per_key(events)])

### Change detection with content hashes

Hash a canonical serialisation of each record (sorted keys, excluding volatile fields such as load timestamps) to
detect which records actually changed between snapshots, without comparing every column. Use a cryptographic hash
(SHA-256) for stability across runs and machines; Python's built-in `hash()` of strings is randomised per process.

In [ ]:
import hashlib
import json

def row_hash(record, exclude=("loaded_at",)):
    canonical = json.dumps({k: v for k, v in record.items() if k not in exclude}, sort_keys=True, default=str, separators=(",", ":"))
    return hashlib.sha256(canonical.encode()).hexdigest()

def diff_snapshots(old, new, key="id"):
    old_h = {r[key]: row_hash(r) for r in old}
    new_h = {r[key]: row_hash(r) for r in new}
    return {
        "inserted": sorted(new_h.keys() - old_h.keys()),
        "deleted": sorted(old_h.keys() - new_h.keys()),
        "updated": sorted(k for k in new_h.keys() & old_h.keys() if new_h[k] != old_h[k]),
    }

old = [{"id": 1, "email": "a@x", "loaded_at": "mon"}, {"id": 2, "email": "b@x", "loaded_at": "mon"}]
new = [{"id": 1, "email": "a@x", "loaded_at": "tue"}, {"id": 2, "email": "b@y", "loaded_at": "tue"}, {"id": 3, "email": "c@x", "loaded_at": "tue"}]
print(row_hash(old[0])[:12], diff_snapshots(old, new))

### Slowly changing dimensions (type 2)

An SCD type 2 dimension keeps history: when an attribute changes, the current row is closed (`valid_to` set,
`is_current` false) and a new current row is inserted. Unchanged records are left alone, which keeps the merge
idempotent.

In [ ]:
def scd2(dim, updates, as_of, key="id", attrs=("email",)):
    out = [dict(r) for r in dim]
    current = {r[key]: r for r in out if r["is_current"]}
    for u in updates:
        row = current.get(u[key])
        if row and all(row[a] == u[a] for a in attrs):
            continue
        if row:
            row.update(valid_to=as_of, is_current=False)
        out.append({key: u[key], **{a: u[a] for a in attrs}, "valid_from": as_of, "valid_to": None, "is_current": True})
    return sorted(out, key=lambda r: (r[key], r["valid_from"]))

dim = [{"id": 1, "email": "a@x", "valid_from": "2026-01-01", "valid_to": None, "is_current": True}]
for r in scd2(dim, [{"id": 1, "email": "a@new"}, {"id": 2, "email": "b@x"}], "2026-11-27"):
    print(r)

## Resilient extraction

### Retries with exponential backoff and jitter

Retry only **transient** failures (timeouts, 429 and 5xx responses, connection resets), never bad requests. Wait
exponentially longer between attempts, cap the wait, and add **jitter** so many clients that failed together do not
retry together (the thundering-herd problem). "Full jitter" picks a uniform random delay between 0 and the capped
exponential value. Inject the random source and the sleep function so the logic is testable.

In [ ]:
import random

def backoff_delays(attempts, base=0.5, cap=30.0, rng=random.random):
    return [round(rng() * min(cap, base * 2 ** i), 3) for i in range(attempts)]

print(backoff_delays(6, rng=lambda: 1.0), "<- the caps without jitter")
print(backoff_delays(6, rng=random.Random(7).random), "<- full jitter")

### Paginating an API with a cursor

APIs return results in pages with a cursor (or `next` URL) for the next page. A generator hides the paging from the
caller, keeps memory flat, and makes it easy to stop early. Guard against a cursor that never changes, which would
loop forever.

In [ ]:
PAGES = {None: {"items": [1, 2], "next": "c2"}, "c2": {"items": [3, 4], "next": "c3"}, "c3": {"items": [5], "next": None}}

def paginate(fetch_page, max_pages=1000):
    cursor, seen = None, set()
    for _ in range(max_pages):
        page = fetch_page(cursor)
        yield from page["items"]
        cursor = page.get("next")
        if cursor is None:
            return
        if cursor in seen:
            raise RuntimeError(f"cursor {cursor!r} repeated")
        seen.add(cursor)
    raise RuntimeError("too many pages")

print(list(paginate(PAGES.__getitem__)))

## Operating pipelines

### Batches with bounded memory and a run summary

Process large inputs in fixed-size batches so memory stays flat, isolate failures to the batch that caused them, and
finish with a **run summary** (counts in, out, rejected, failed batches, duration) that is logged and stored. The
summary is what alerting and reconciliation read.

In [ ]:
import time
from itertools import batched

def run(rows, size, handle):
    summary = {"batches": 0, "rows_in": 0, "rows_out": 0, "failed_batches": []}
    start = time.perf_counter()
    for number, batch in enumerate(batched(rows, size), 1):
        summary["batches"] += 1
        summary["rows_in"] += len(batch)
        try:
            summary["rows_out"] += handle(batch)
        except Exception as e:
            summary["failed_batches"].append((number, type(e).__name__))
    summary["seconds"] = round(time.perf_counter() - start, 3)
    return summary

def handle(batch):
    if 13 in batch:
        raise ValueError("poison record")
    return len(batch)

print(run(range(25), 10, handle))

### Backfills and reconciliation

A **backfill** reprocesses a historical range: plan it as a list of partition chunks, skip partitions already done,
and run chunks independently so a failure can be resumed. **Reconciliation** compares counts (or sums) between source
and target per partition after a load, and alerts on any mismatch beyond a tolerance.

In [ ]:
from datetime import date

def backfill_plan(start, end, done, chunk_days=7):
    chunks, current = [], []
    for i in range((end - start).days + 1):
        day = start + timedelta(days=i)
        if day in done:              # a finished partition ends the current chunk
            chunks, current = chunks + [current] if current else chunks, []
            continue
        current.append(day)
        if len(current) == chunk_days:
            chunks, current = chunks + [current], []
    chunks += [current] if current else []
    return [(c[0].isoformat(), c[-1].isoformat()) for c in chunks]

print(backfill_plan(date(2026, 11, 1), date(2026, 11, 20), done={date(2026, 11, 3), date(2026, 11, 4)}))

def reconcile(source, target, tolerance=0):
    keys = sorted(source.keys() | target.keys())
    return [(k, source.get(k, 0), target.get(k, 0)) for k in keys if abs(source.get(k, 0) - target.get(k, 0)) > tolerance]

print(reconcile({"2026-11-26": 1180, "2026-11-27": 950}, {"2026-11-26": 1180, "2026-11-27": 948, "2026-11-28": 3}))

### Pseudonymising personal data

Replace direct identifiers (emails, phone numbers) with a **keyed hash** (HMAC-SHA256 with a secret key held in a
secrets manager) so records can still be joined on the token, but the raw value cannot be recovered or looked up with a
rainbow table. Normalise before hashing (trim, lower-case emails) so the same person always maps to the same token.

In [ ]:
import hmac

def pseudonymise(value, key):
    normalised = value.strip().lower().encode()
    return hmac.new(key, normalised, hashlib.sha256).hexdigest()[:16]

key = b"load-from-a-secrets-manager"
print(pseudonymise(" Alice@Example.com", key), pseudonymise("alice@example.com", key))

**Next:** practise these entries in [Python practice 12: data engineering patterns](../practice/python-practice-12-data-engineering-patterns.ipynb).